# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


The question "which pages first?" sounds like ranking on the surface, but a ranking score needs something to be right about underneath it. I'm framing the modeling task as **binary classification**: predict whether a page's search traffic is currently declining (`is_declining_label`), using signals available about that page. I then take the model's predicted **probability** of decline for every page and sort by it — that sorted list is the ranked review queue an editor works from.

So: classification model → ranking product. The task type is classification; the deliverable the editor sees is a ranking.

In [2]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")
print(f"Unique clients: {df['client_id'].nunique()}")

Rows: 30000
Columns: 44
Unique clients: 32


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `is_declining_label`, defined in the data as `trend_direction == "down"` (impressions dropped more than 20% over the last 30 days vs. the prior 30 days). This is an **observed outcome** — a measured before/after change in real traffic — not an editor's opinion invented after the fact.

**Leakage trap to respect:** `trend_direction` and `trend_pct` are the columns this label is *built from*, so they can never be model features — a model that saw them would just be re-deriving its own label.

**Being honest about the proxy gap:** "impressions declining" is not the same thing as "needs a content refresh." A page can decline from seasonality or a SERP change, not stale content. That gap is why this stays decision-support, not an auto-trigger.

In [3]:
# Build the target exactly the way the data dictionary defines it.
# trend_direction / trend_pct are the SOURCE of this label -> never features.
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print("Class balance (is_declining_label):")
print(df['is_declining_label'].value_counts(normalize=True).round(3))

Class balance (is_declining_label):
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Two metrics, for two audiences:

- **Model quality — ROC-AUC**, on a held-out split by `client_id` (never mix one client's pages across train/test). AUC says whether the model separates decliners from non-decliners better than chance.
- **Business metric — precision@K**, e.g. "of the top 50 pages flagged this week, how many are actually declining?" That's the number an editor feels, since they only review a fixed number of pages per week.

**Baseline to beat:** always predicting "declining" already gets ~54% accuracy for free (below) and AUC = 0.50 by definition. Both metrics need to clear those by a real margin.

In [4]:
majority_class_rate = df['is_declining_label'].mean()
print(f"Majority-class baseline (predict 'declining' for everyone): {majority_class_rate:.1%} accuracy, AUC = 0.50")

Majority-class baseline (predict 'declining' for everyone): 54.2% accuracy, AUC = 0.50


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page** (`content_id`), owned by one `client_id`, with metrics aggregated over a trailing 90-day window. `content_id` and `client_id` are pseudonyms — used only for grouping/splits, never as features.

In [5]:
unit_of_analysis_cols = [
    'content_id', 'client_id',
    'impressions_90d', 'ctr', 'avg_position',
    'content_age_days', 'days_since_last_update', 'engagement_rate',
    'is_declining_label',
]

print(f"Rows: {len(df)}  |  Unique pages: {df['content_id'].nunique()}")
df[unit_of_analysis_cols].head(5)

Rows: 30000  |  Unique pages: 30000


,content_id,client_id,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update,engagement_rate,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,0.76,10.6,187,20,5.88,1
1,content_a1fb4e703a9e,client_4e07408562,15320,0.05,20.3,445,25,0.00,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,0.09,36.5,141,20,0.00,1
3,content_331d6c4de07b,client_19581e27de,11751,0.49,6.2,463,22,1.28,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,0.13,44.0,263,14,0.00,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule would look like "flag anything with `avg_position` worse than X." To test whether one signal alone could do the job, I checked how strongly each candidate feature correlates with the label on its own (code below). Every single signal correlates weakly — the strongest, `content_age_days`, is only -0.16. No individual signal is a usable threshold rule on its own.

But a pattern can still be real even when no single signal is strong — it can be spread thinly across several weak, interacting signals at once. That combination is what a model can learn and an if-statement can't hand-write. That's the case for ML over a rule here.

In [6]:
candidate_features = ['avg_position', 'ctr', 'content_age_days', 'days_since_last_update', 'engagement_rate', 'impressions_90d']

correlations = (
    df[candidate_features + ['is_declining_label']]
    .dropna()
    .corr()['is_declining_label']
    .drop('is_declining_label')
    .sort_values(key=abs, ascending=False)
)
print("Correlation of each single signal with is_declining_label:")
print(correlations.round(3))

Correlation of each single signal with is_declining_label:
content_age_days         -0.164
days_since_last_update    0.081
ctr                      -0.062
avg_position             -0.029
impressions_90d          -0.018
engagement_rate          -0.013
Name: is_declining_label, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.